# 99 — Final Submission Check

يفحص هذا الدفتر **الملفات الفعلية الموجودة** في مجلد المشروع (`ROOT`) فقط، باستخدام دوال التحقق الموجودة في `scripts/day5_delivery.py` (`evidence_status` و`validate_submission` و`reflection_check` و`replay_submission` و`verify_bundle`) ومنطق قائمة الملفات من خلية Day 5 الأخيرة.
أي ملف أو دليل غير موجود يظهر `MISSING`؛ لا يُنشأ أي ملف ولا تُحسب أي نتيجة بديلة.
خلية الإعداد وخلية سرد ZIPs منسوختان حرفيًا من Master Notebook (Day 5).

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
PREVIOUS_SUPPORT = {
 "scripts/day2_validation.py": ("1e1da4acbee8941bef07245b259fb6c27ced4ad7", "d2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43"),
 "scripts/day3_decision.py": ("c2dc52e129878d0d487db1874891c658de7c4d0d", "40f176633c91fc530db222978bc44495a401b888b1ce448728948f7ae0ca9e12"),
 "scripts/day4_trust.py": ("4f6892d5c02923b5f8e3c78f4fcead73b043570f", "7e0f54bdbed94b28d09cddd02ce0ba0471d5046f05c7b70589e3e4be9902f20c")}
for relative, (revision, digest) in PREVIOUS_SUPPORT.items():
    fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{revision}/{relative}", ROOT / relative, digest)
LAB_REVISION = "f486fc50dd9ac8403016facc58cf6a62beb4abf4"
SUPPORT_HASHES = {'scripts/day5_final.py': '71553cffc66f972e77d470a347cae40e4ca2802026535b48cb40168a59362b10', 'scripts/day5_delivery.py': 'eba5b5820550b44ccca2cf7f5d56a826c015de552e5db8d7e4c27a6e956ecd21', 'scripts/inference.py': 'abf1af0bca47bcd08e66f8218276773d39083e6f42f42caeea5d8b2db70c0a6a', 'scripts/replay_final.py': 'adcb2e40aff819439068cd7a2aca9fcb6f8632b0c551a9d04f7bd63efc8506fb', 'scripts/rebuild_final.py': 'ac18b78160dbc33b720f3e1038667110760832c095e23c1ea7f909d4c5760dfd', 'data/tamweel_oof_matrix.csv': 'f8c9429c7eed015d53978fba93b4b32ed3266fa15b63471b03881ab5c09804b5', 'data/day5_oof_example.json': '9edcb173ab5aa504af3ef79c4f3371cd285fa4b171be1bcc6dd9da1d7fb0394d', 'presentation/FINAL_PRESENTATION_TEMPLATE.md': '8d3df570727b18db097c9c78ed518ac09bd8766dbdba2817ae8883e43ed5de39', 'presentation/FINAL_PRESENTATION_GUIDE.md': '1d24149ef2e56df0a6c44e000157135904b8ba3d252445e4367a6de547c87b19'}
for relative, digest in SUPPORT_HASHES.items():
    fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/{relative}", ROOT / relative, digest)
print("Day 5 setup verified | إعداد اليوم الخامس جاهز")

Package                 Version
numpy                   2.1.3
pandas                  2.2.3
scipy                   1.16.3
scikit-learn            1.6.1
matplotlib              3.10.0
xgboost                 3.4.1
lightgbm                4.6.0
shap                    0.52.0
numba                   0.61.2
optuna                  4.5.0
Python 3.13.16 | CPU | seed=211 | n_jobs=2 | FAST_MODE=True
Verified course files. Next: run the data and compatibility checks.
Day 5 setup verified | إعداد اليوم الخامس جاهز


In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from day3_decision import CostPolicy
from day5_delivery import (RESPONSES, reflection_check, evidence_status,
    validate_submission, replay_submission, verify_bundle)

check_log = []
day5_checkpoint = None

In [ ]:
from pathlib import Path

for day in range(1, 5):
    matches = list(ROOT.glob(f"*day{day}*artifacts*.zip")) + list(ROOT.glob(f"day{day}_artifacts.zip"))
    print(f"Day {day}:", [p.name for p in matches])

Day 1: []
Day 2: []
Day 3: []
Day 4: []


In [ ]:
completeness=evidence_status(ROOT)
display(pd.DataFrame(completeness)[['item','status']])
check_log += [(row['item'], row['status']) for row in completeness]

In [ ]:
required_files = ["PROJECT_README.md","reports/MODEL_CARD.md","reports/ENSEMBLE_DECISION.md","submission/submission.csv",
    "requirements-colab.txt","constraints.txt","data/data_contract.json","data/data_manifest.json","data/tamweel_train.csv","data/tamweel_challenge.csv",
    "presentation/FINAL_PRESENTATION_TEMPLATE.md","presentation/FINAL_PRESENTATION_GUIDE.md",*PREVIOUS_SUPPORT.keys(),*[p for p in SUPPORT_HASHES if p.startswith('scripts/')]]
tables_names = ["day5_roles.csv","day5_oof_predictions.csv","day5_fold_scores.csv","ensemble_comparison.csv",
    "day5_threshold_sweep.csv","day5_region_audit.csv","day5_period_capacity.csv","day5_cost_sensitivity.csv"]
artifact_names=[*tables_names,"day5_probability_correlation.csv","day5_residual_correlation.csv","day5_run.json","day5_reflection.json",
    "day5_oof_provenance.json","day5_ensemble_gate.json","final_policy.json","final_metrics.json","day5_final_provenance.json",
    "day5_calibration_predictions.csv","day5_calibration_fit_bins.csv","day5_project_check.json","environment.json",
    "day5_diversity.png","day5_ensemble_comparison.png","day5_policy_regions.png","day5_calibration_fit.png","day5_challenge_capacity.png"]
required_files += ["artifacts/"+name for name in artifact_names]
required_files.append("artifacts/final_model/model_manifest.json")
file_rows = [{"item": name, "status": "PRESENT_REQUIRES_REVIEW" if (ROOT/name).is_file() and (ROOT/name).stat().st_size>0 else "MISSING"}
             for name in required_files]
file_table = pd.DataFrame(file_rows)
display(file_table)
n_missing = int((file_table.status == "MISSING").sum())
print(f"Required project files: {len(file_table)-n_missing} present, {n_missing} MISSING (presence only; content is not verified).")
check_log.append(("required_files", f"{n_missing} MISSING of {len(file_table)}"))

In [ ]:
submission_path = ROOT / "submission/submission.csv"
policy_path = ROOT / "artifacts/final_policy.json"
challenge_path = ROOT / "data/tamweel_challenge.csv"
if submission_path.is_file() and policy_path.is_file() and challenge_path.is_file():
    final_policy = json.loads(policy_path.read_text(encoding="utf-8"))
    batch_audit = validate_submission(pd.read_csv(submission_path), pd.read_csv(challenge_path),
        final_policy["calibrated_threshold"], CostPolicy(**final_policy["cost_policy"]))
    display(pd.DataFrame([batch_audit]))
    check_log.append(("validate_submission", "PASSED"))
else:
    absent = [str(p.relative_to(ROOT)) for p in (submission_path, policy_path, challenge_path) if not p.is_file()]
    print("MISSING — validate_submission not run; absent:", absent)
    check_log.append(("validate_submission", "MISSING"))

In [ ]:
reflection_path = ROOT / "artifacts/day5_reflection.json"
run_path = ROOT / "artifacts/day5_run.json"
if reflection_path.is_file() and run_path.is_file():
    saved_reflection = json.loads(reflection_path.read_text(encoding="utf-8"))
    saved_run = json.loads(run_path.read_text(encoding="utf-8"))
    day5_checkpoint = reflection_check(saved_reflection["responses"], saved_run["source"])
    print(day5_checkpoint["status"], "| Missing:", day5_checkpoint["missing"])
    check_log.append(("reflection_check", day5_checkpoint["status"]))
else:
    print("MISSING — reflection_check not run; day5_reflection.json / day5_run.json not found in artifacts/.")
    check_log.append(("reflection_check", "MISSING"))

In [ ]:
if submission_path.is_file() and policy_path.is_file() and challenge_path.is_file() and (ROOT / "artifacts/final_model").is_dir():
    submission = pd.read_csv(submission_path)
    reproduced=replay_submission(ROOT)
    assert np.allclose(reproduced.probability,submission.probability,rtol=0,atol=1e-12)
    assert np.array_equal(reproduced.decision,submission.decision)
    print("REPLAY_MATCH | saved submission reproduced from artifacts/final_model")
    check_log.append(("replay_submission", "REPLAY_MATCH"))
else:
    print("MISSING — replay_submission not run; submission, final_policy.json, challenge data or artifacts/final_model absent.")
    check_log.append(("replay_submission", "MISSING"))

In [ ]:
bundle_path = ROOT / "submission/project_bundle.zip"
if bundle_path.is_file():
    bundle_check = verify_bundle(bundle_path)
    print(bundle_check)
    check_log.append(("verify_bundle", bundle_check["status"]))
else:
    print("MISSING — submission/project_bundle.zip not found; verify_bundle not run.")
    check_log.append(("verify_bundle", "MISSING"))

In [ ]:
full_ready = day5_checkpoint is not None and day5_checkpoint['status']=="READY_FOR_REVIEW" and all(r['status'] in ("READY_FOR_REVIEW","PRESENT_REQUIRES_REVIEW") for r in completeness)
full_status = "READY_FOR_HUMAN_REVIEW" if full_ready else "PROJECT_WORK_REQUIRED"
display(pd.DataFrame(check_log, columns=["item", "status"]))
print(full_status)